# Lab 03: Advanced DataFrame Operations

## Objectives
- Perform different types of joins (inner, outer, left, right, semi, anti)
- Use window functions for analytics (ranking, lead/lag, aggregations)
- Work with complex data types (arrays, maps, structs)
- Create and use User Defined Functions (UDFs)
- Optimize join performance

## Domain Coverage
- **DataFrame API — 30%**

## Estimates
- **Time:** ~90-120 minutes
- **Difficulty:** Intermediate
- **Prerequisites:** Lab 01 (Spark Fundamentals), Lab 02 (Data Loading & Transformation)

![Advanced Operations Diagram](../assets/diagrams/lab-03-advanced-operations.mmd)

In [ ]:
# Import required libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, avg, sum, max, min
from pyspark.sql.functions import row_number, rank, dense_rank, lag, lead
from pyspark.sql.functions import explode, array, struct, map_from_arrays, map_keys, map_values
from pyspark.sql.functions import udf, pandas_udf
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, ArrayType
from pyspark.sql.types import MapType
from pyspark.sql.window import Window
import pandas as pd

In [ ]:
# Create Spark session
spark = SparkSession.builder \
    .appName("AdvancedDataFrameOperations") \
    .master("spark://spark-master:7077") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

# Verify Spark session
print(f"Spark version: {spark.version}")
print(f"Spark master: {spark.conf.get('spark.master')}")

## A. Creating Sample Data for Joins

Let's create two DataFrames to practice different join operations.

In [ ]:
# Create employees DataFrame
employees_data = [
    (1, "Alice", "Engineering", 75000),
    (2, "Bob", "Marketing", 65000),
    (3, "Charlie", "Engineering", 85000),
    (4, "Diana", "Sales", 60000),
    (5, "Eve", "Marketing", 70000),
    (6, "Frank", None, 55000)  # Employee without department
]

employees_df = spark.createDataFrame(employees_data, ["id", "name", "department", "salary"])
print("Employees DataFrame:")
employees_df.show()

In [ ]:
# Create departments DataFrame
departments_data = [
    ("Engineering", "Building A", 100),
    ("Marketing", "Building B", 50),
    ("Sales", "Building C", 75),
    ("HR", "Building D", 25),  # Department without employees
    ("Finance", "Building E", 30)  # Department without employees
]

departments_df = spark.createDataFrame(departments_data, ["department", "location", "budget"])
print("Departments DataFrame:")
departments_df.show()

## B. Inner Join

Inner join returns only rows with matching keys in both DataFrames.

In [ ]:
# Inner join
inner_join = employees_df.join(departments_df, "department", "inner")

print("Inner Join (only matching departments):")
inner_join.show()

print(f"\nEmployees: {employees_df.count()}")
print(f"Departments: {departments_df.count()}")
print(f"Inner join result: {inner_join.count()}")

## C. Left Join

Left join returns all rows from the left DataFrame and matching rows from the right DataFrame.

In [ ]:
# Left join
left_join = employees_df.join(departments_df, "department", "left")

print("Left Join (all employees, matching departments):")
left_join.show()

print(f"\nLeft join result: {left_join.count()}")

## D. Right Join

Right join returns all rows from the right DataFrame and matching rows from the left DataFrame.

In [ ]:
# Right join
right_join = employees_df.join(departments_df, "department", "right")

print("Right Join (all departments, matching employees):")
right_join.show()

print(f"\nRight join result: {right_join.count()}")

## E. Outer Join

Outer join returns all rows from both DataFrames, with nulls where there's no match.

In [ ]:
# Outer join
outer_join = employees_df.join(departments_df, "department", "outer")

print("Outer Join (all rows from both DataFrames):")
outer_join.show()

print(f"\nOuter join result: {outer_join.count()}")

## F. Left Semi Join

Left semi join filters the left DataFrame to keep only rows with matches in the right DataFrame.

In [ ]:
# Left semi join
left_semi_join = employees_df.join(departments_df, "department", "leftsemi")

print("Left Semi Join (employees with matching departments):")
left_semi_join.show()

print(f"\nLeft semi join result: {left_semi_join.count()}")

## G. Left Anti Join

Left anti join filters the left DataFrame to keep only rows without matches in the right DataFrame.

In [ ]:
# Left anti join
left_anti_join = employees_df.join(departments_df, "department", "leftanti")

print("Left Anti Join (employees without matching departments):")
left_anti_join.show()

print(f"\nLeft anti join result: {left_anti_join.count()}")

## H. Window Functions

Window functions perform calculations across a set of rows related to the current row.

In [ ]:
# Define window specification
window_spec = Window.partitionBy("department").orderBy(col("salary").desc())

# Ranking functions
employees_with_rank = employees_df.withColumn("row_num", row_number().over(window_spec)) \
                                      .withColumn("rank", rank().over(window_spec)) \
                                      .withColumn("dense_rank", dense_rank().over(window_spec))

print("Window Functions - Ranking:")
employees_with_rank.show()

In [ ]:
# Analytic functions (lag/lead)
employees_with_lag = employees_df.withColumn("lag_salary", lag("salary", 1).over(window_spec)) \
                                  .withColumn("lead_salary", lead("salary", 1).over(window_spec))

print("Window Functions - Lag/Lead:")
employees_with_lag.show()

In [ ]:
# Aggregate functions over window
window_agg_spec = Window.partitionBy("department")

employees_with_agg = employees_df.withColumn("dept_avg_salary", avg("salary").over(window_agg_spec)) \
                                 .withColumn("dept_total_salary", sum("salary").over(window_agg_spec)) \
                                 .withColumn("dept_max_salary", max("salary").over(window_agg_spec))

print("Window Functions - Aggregates:")
employees_with_agg.show()

## I. Complex Data Types

Spark supports complex data types like arrays, maps, and structs.

In [ ]:
# Array operations
employees_with_skills = employees_df.withColumn("skills", array("Python", "Spark", "SQL"))

print("Array column added:")
employees_with_skills.select("name", "skills").show(truncate=False)

# Explode array to rows
from pyspark.sql.functions import explode
exploded_skills = employees_with_skills.withColumn("skill", explode(col("skills")))

print("\nExploded array to rows:")
exploded_skills.select("name", "skill").show()

In [ ]:
# Map operations
employees_with_map = employees_df.withColumn(
    "metadata",
    map_from_arrays(array("hire_date", "manager"), array("2020-01-01", "Alice"))
)

print("Map column added:")
employees_with_map.select("name", "metadata").show(truncate=False)

# Access map keys and values
map_ops = employees_with_map.withColumn("map_keys", map_keys(col("metadata"))) \
                              .withColumn("map_values", map_values(col("metadata")))

print("\nMap keys and values:")
map_ops.select("name", "map_keys", "map_values").show(truncate=False)

In [ ]:
# Struct operations
employees_with_struct = employees_df.withColumn(
    "address",
    struct("name", "department").alias("address")
)

print("Struct column added:")
employees_with_struct.select("name", "address").show(truncate=False)

# Access struct fields
struct_ops = employees_with_struct.select(col("name"), col("address.department"))

print("\nAccess struct fields:")
struct_ops.show(truncate=False)

## J. User Defined Functions (UDFs)

UDFs allow you to define custom functions for data transformation.

In [ ]:
# Define a Python UDF
def salary_category(salary):
    if salary < 60000:
        return "Low"
    elif salary < 80000:
        return "Medium"
    else:
        return "High"

# Register UDF
salary_category_udf = udf(salary_category, StringType())

# Apply UDF
employees_with_category = employees_df.withColumn(
    "salary_category",
    salary_category_udf(col("salary"))
)

print("Python UDF applied:")
employees_with_category.select("name", "salary", "salary_category").show()

In [ ]:
# Define a Pandas UDF (vectorized UDF)
@pandas_udf(DoubleType())
def calculate_bonus_pandas(salary: pd.Series) -> pd.Series:
    return salary * 0.10

# Apply Pandas UDF
employees_with_pandas_bonus = employees_df.withColumn(
    "bonus",
    calculate_bonus_pandas(col("salary"))
)

print("Pandas UDF applied:")
employees_with_pandas_bonus.select("name", "salary", "bonus").show()

## K. Join Performance Optimization

Optimize join performance using broadcast joins for small tables.

In [ ]:
from pyspark.sql.functions import broadcast

# Broadcast join (for small tables)
broadcast_join = employees_df.join(broadcast(departments_df), "department")

print("Broadcast join completed:")
broadcast_join.show()

# Explain the plan to see broadcast
print("\nQuery plan (look for BroadcastHashJoin):")
broadcast_join.explain()

## Exam-Style Review

**Q1.** What is the difference between left semi join and left anti join?
- A) Left semi keeps matches, left anti keeps non-matches
- B) Left semi keeps non-matches, left anti keeps matches
- C) They are identical
- D) Left semi is for outer joins, left anti is for inner joins

**Q2.** Which window function assigns unique sequential numbers without gaps?
- A) row_number()
- B) rank()
- C) dense_rank()
- D) count()

**Q3.** What is the advantage of Pandas UDFs over regular UDFs?
- A) Pandas UDFs are easier to write
- B) Pandas UDFs use vectorized operations for better performance
- C) Pandas UDFs support more data types
- D) Pandas UDFs are the default in Spark

**Q4.** When should you use a broadcast join?
- A) When joining two large DataFrames
- B) When one DataFrame is small enough to fit in memory
- C) When you need to preserve all rows
- D) When you need to sort the results

### Answers
- **Q1: A** — Left semi join keeps rows with matches, left anti join keeps rows without matches.
- **Q2: A** — row_number() assigns unique sequential numbers without gaps, even for ties.
- **Q3: B** — Pandas UDFs use vectorized operations (Apache Arrow) for better performance than row-by-row UDFs.
- **Q4: B** — Use broadcast joins when one DataFrame is small enough to fit in memory, as it avoids shuffling the small table.

## Key Takeaways
- Different join types serve different purposes (inner, left, right, outer, semi, anti)
- Window functions enable analytics without reducing row count (ranking, lag/lead, aggregates)
- Complex data types (arrays, maps, structs) handle nested and semi-structured data
- UDFs allow custom transformations, with Pandas UDFs offering better performance
- Broadcast joins optimize performance when one table is small
- Choose the right join type based on your data requirements

**Next:** [Lab 04 — Spark SQL & Optimization](chapter-04-spark-sql-optimization.ipynb)

In [ ]:
# Cleanup
spark.catalog.clearCache()
print("Lab 03 complete. Cache cleared.")